# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Method choice

I use **K-Means clustering** because my lane is Structured Content Archetype Clustering and there is no ground-truth archetype label in the warehouse. The goal is to group content pages with similar performance patterns across impressions, clicks, CTR, average search position, and position volatility.

The feature distributions are strongly skewed, especially impressions and clicks, so non-negative count features will be log-transformed before standardization. Standardization is then used so that features measured on different scales do not dominate K-Means distance calculations.

I will evaluate candidate cluster counts using silhouette score, cluster size, stability, and interpretability. Cluster names will be assigned only after inspecting the resulting cluster profiles; they are descriptive decision-support labels, not ground-truth classes.

The model is intended to discover performance archetypes that can support actions such as protect, improve, rewrite, merge, prune, or monitor. It does not establish causal relationships between page characteristics and performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )


HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("Using March 2026 development data.")

DuckDB connection established.
Using March 2026 development data.


In [3]:
# Build the same five-feature vector established during ML-05

feature_vector = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

clustering_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]

print("Rows:", len(feature_vector))
print("Clustering features:", clustering_features)

display(feature_vector.head())

print("\nMissing values:")
display(
    feature_vector[clustering_features]
    .isna()
    .sum()
    .to_frame("missing_rows")
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Clustering features: ['impressions', 'clicks', 'ctr', 'avg_position', 'position_volatility']


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,0.111235,5.908100,5.911676
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,0.000000,6.419872,6.156393
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,0.000000,5.177774,2.109420
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,0.129534,4.685335,1.141410
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0,0.000000,5.333333,1.984063



Missing values:


,missing_rows
impressions,0
clicks,0
ctr,0
avg_position,1434
position_volatility,15181


### Split design

Because this is an unsupervised clustering problem with no ground-truth target, a conventional supervised train/test split is not appropriate. I use a client-grouped development/holdout design: approximately 80% of clients are used for model development and the remaining 20% are held out for stability evaluation.

Grouping by client prevents pages from the same client appearing in both groups, which reduces the risk that client-specific performance patterns make the clustering look more stable than it is. The March 2026 slice is kept fixed so the model remains comparable with the Week-4 baseline.

The holdout is used to assess whether the learned preprocessing and cluster structure transfer reasonably to unseen clients; it is not treated as a supervised prediction test.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

# Use the client ID only for grouping, never as a clustering feature.
groups = feature_vector["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, holdout_idx = next(
    splitter.split(
        feature_vector,
        groups=groups
    )
)

model_df = feature_vector.iloc[train_idx].copy()
holdout_df = feature_vector.iloc[holdout_idx].copy()

print("Development rows:", len(model_df))
print("Holdout rows:", len(holdout_df))

print(
    "Development clients:",
    model_df["client_hash_id"].nunique()
)

print(
    "Holdout clients:",
    holdout_df["client_hash_id"].nunique()
)

print(
    "Client overlap:",
    len(
        set(model_df["client_hash_id"])
        &
        set(holdout_df["client_hash_id"])
    )
)

Development rows: 138310
Holdout rows: 38428
Development clients: 37
Holdout clients: 10
Client overlap: 0


### Model training and comparison

I use K-Means on the development clients and evaluate candidate values of K using silhouette score and cluster-size balance. The preprocessing is fitted only on the development clients and then applied unchanged to the holdout clients.

Because this lane is unsupervised, there is no ground-truth target that supports a supervised precision or accuracy comparison. Therefore, the Week-4 baseline is treated as a transparent review-priority benchmark. I compare the clustering structure with the baseline opportunity scores to understand whether the discovered groups separate pages with different levels of review priority.

The model is considered useful only if its clusters are reasonably separated, stable on unseen clients, interpretable, and actionable. A higher-complexity clustering solution is not preferred merely because it produces more clusters.

In [8]:
# Prepare development and holdout feature matrices

model_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]

X_train = model_df[model_features].copy()
X_holdout = holdout_df[model_features].copy()

# Count features are highly right-skewed.
# Log-transform only the non-negative count variables.
count_features = [
    "impressions",
    "clicks"
]

for col in count_features:
    X_train[col] = np.log1p(X_train[col])
    X_holdout[col] = np.log1p(X_holdout[col])

# Fit preprocessing ONLY on development data.
preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

X_train_scaled = preprocessor.fit_transform(X_train)
X_holdout_scaled = preprocessor.transform(X_holdout)

print("Development matrix:", X_train_scaled.shape)
print("Holdout matrix:", X_holdout_scaled.shape)

Development matrix: (138310, 5)
Holdout matrix: (38428, 5)


In [9]:
# Evaluate candidate numbers of clusters

k_results = []

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    train_labels = kmeans.fit_predict(X_train_scaled)

    silhouette = silhouette_score(
        X_train_scaled,
        train_labels,
        sample_size=20000,
        random_state=42
    )

    cluster_sizes = pd.Series(train_labels).value_counts()

    k_results.append(
        {
            "k": k,
            "silhouette": silhouette,
            "smallest_cluster": cluster_sizes.min(),
            "largest_cluster": cluster_sizes.max(),
            "smallest_cluster_pct":
                100 * cluster_sizes.min() / len(train_labels)
        }
    )

k_results_df = pd.DataFrame(k_results)

display(
    k_results_df.sort_values(
        "silhouette",
        ascending=False
    )
)

,k,silhouette,smallest_cluster,largest_cluster,smallest_cluster_pct
2,4,0.410161,303,74520,0.219073
1,3,0.401746,27192,74719,19.660184
6,8,0.389395,151,39620,0.109175
5,7,0.387997,151,46776,0.109175
4,6,0.382722,303,46653,0.219073
3,5,0.372462,303,48158,0.219073
0,2,0.335095,40240,98070,29.094064


### Final cluster count

I select **K=3** for the final clustering solution.

Although K=4 has the highest development-set silhouette score (0.4102), it produces a very small cluster containing only 303 observations (0.22% of the development data). K=3 has a similar silhouette score (0.4017) while maintaining substantially better cluster-size balance, with the smallest cluster containing 19.66% of observations.

I therefore prefer K=3 because the small difference in separation does not justify the highly fragmented cluster structure produced by K=4. This choice favors interpretability and usefulness for content archetype analysis rather than maximizing a single metric.

In [10]:
# Fit the final K=3 clustering model
final_k = 3

final_kmeans = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

train_labels = final_kmeans.fit_predict(X_train_scaled)
holdout_labels = final_kmeans.predict(X_holdout_scaled)

print("Final K:", final_k)
print("Development cluster sizes:")
print(pd.Series(train_labels).value_counts().sort_index())

print("\nHoldout cluster sizes:")
print(pd.Series(holdout_labels).value_counts().sort_index())

Final K: 3
Development cluster sizes:
0    27192
1    36399
2    74719
Name: count, dtype: int64

Holdout cluster sizes:
0     9810
1    13321
2    15297
Name: count, dtype: int64


In [11]:
# Evaluate the final clustering on the unseen-client holdout
train_silhouette = silhouette_score(
    X_train_scaled,
    train_labels,
    sample_size=20000,
    random_state=42
)

holdout_silhouette = silhouette_score(
    X_holdout_scaled,
    holdout_labels,
    sample_size=min(20000, len(X_holdout_scaled)),
    random_state=42
)

print(f"Development silhouette: {train_silhouette:.4f}")
print(f"Holdout silhouette:     {holdout_silhouette:.4f}")

Development silhouette: 0.4017
Holdout silhouette:     0.3907


In [12]:
# Profile the final K=3 clusters using the original, untransformed features

profile_df = model_df[[
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]].copy()

profile_df["cluster"] = train_labels

cluster_profile = (
    profile_df
    .groupby("cluster")
    .agg(
        pages=("content_hash_id", "count"),
        clients=("client_hash_id", "nunique"),
        median_impressions=("impressions", "median"),
        median_clicks=("clicks", "median"),
        median_ctr=("ctr", "median"),
        median_avg_position=("avg_position", "median"),
        median_position_volatility=("position_volatility", "median"),
        mean_impressions=("impressions", "mean"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

cluster_profile["page_share_pct"] = (
    100 * cluster_profile["pages"] / len(profile_df)
)

display(
    cluster_profile.sort_values("cluster")
)

,cluster,pages,clients,median_impressions,median_clicks,median_ctr,median_avg_position,median_position_volatility,mean_impressions,mean_ctr,page_share_pct
0,0,27192,36,59.0,0.0,0.000000,40.500000,22.582873,169.280928,0.136987,19.660184
1,1,36399,27,2460.0,6.0,0.282885,7.917966,3.400076,4983.169648,0.425394,26.316969
2,2,74719,37,38.0,0.0,0.000000,7.500000,4.387071,209.487520,0.643997,54.022847


In [13]:
# Compare each cluster against the overall development-set median

overall_median = profile_df[
    model_features
].median()

comparison = cluster_profile[[
    "cluster",
    "pages",
    "page_share_pct",
    "median_impressions",
    "median_clicks",
    "median_ctr",
    "median_avg_position",
    "median_position_volatility"
]].copy()

comparison["impressions_vs_overall"] = (
    comparison["median_impressions"] / overall_median["impressions"]
)

comparison["ctr_vs_overall"] = (
    comparison["median_ctr"] / overall_median["ctr"]
    if overall_median["ctr"] > 0 else np.nan
)

comparison["position_vs_overall"] = (
    comparison["median_avg_position"] / overall_median["avg_position"]
)

display(comparison)

,cluster,pages,page_share_pct,median_impressions,median_clicks,median_ctr,median_avg_position,median_position_volatility,impressions_vs_overall,ctr_vs_overall,position_vs_overall
0,0,27192,19.660184,59.0,0.0,0.000000,40.500000,22.582873,0.433824,NaN,4.432836
1,1,36399,26.316969,2460.0,6.0,0.282885,7.917966,3.400076,18.088235,NaN,0.866643
2,2,74719,54.022847,38.0,0.0,0.000000,7.500000,4.387071,0.279412,NaN,0.820896


### Baseline opportunity comparison

The Week-4 baseline is used here as a transparent review-priority benchmark rather than as a competing supervised model. Because the clustering task has no ground-truth archetype labels, there is no valid accuracy or precision metric that can directly compare K-Means with the baseline.

Instead, I examine how the baseline opportunity score is distributed across the discovered clusters. This shows whether the multidimensional archetypes correspond to different levels of review priority identified by the simpler Week-4 rule.

The baseline score is calculated using the same March 2026 search-performance fields and the same position-aware CTR-gap logic used in Week 4. The baseline score is not used as an input to K-Means.

In [14]:
# Recreate the Week-4 baseline opportunity score on the development set.
# The baseline score is NOT used as a clustering feature.

baseline_df = model_df[[
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "ctr",
    "avg_position"
]].copy()

def position_bucket(position):
    if pd.isna(position):
        return "missing"
    if position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    else:
        return "20+"

baseline_df["position_bucket"] = baseline_df["avg_position"].apply(
    position_bucket
)

# Expected CTR is learned from the development data only.
expected_ctr = (
    baseline_df[
        (baseline_df["impressions"] >= 500) &
        (baseline_df["position_bucket"] != "missing")
    ]
    .groupby("position_bucket")["ctr"]
    .median()
)

baseline_df["expected_ctr"] = baseline_df["position_bucket"].map(
    expected_ctr
)

baseline_df["ctr_gap"] = (
    baseline_df["expected_ctr"] - baseline_df["ctr"]
).clip(lower=0)

baseline_df["baseline_score"] = np.where(
    baseline_df["impressions"] >= 500,
    np.log1p(baseline_df["impressions"]) * baseline_df["ctr_gap"],
    0
)

baseline_df["cluster"] = train_labels

baseline_cluster_summary = (
    baseline_df
    .groupby("cluster")
    .agg(
        pages=("content_hash_id", "count"),
        median_baseline_score=("baseline_score", "median"),
        mean_baseline_score=("baseline_score", "mean"),
        pages_with_opportunity=("baseline_score", lambda x: (x > 0).sum())
    )
    .reset_index()
)

baseline_cluster_summary["opportunity_rate_pct"] = (
    100
    * baseline_cluster_summary["pages_with_opportunity"]
    / baseline_cluster_summary["pages"]
)

display(baseline_cluster_summary)

,cluster,pages,median_baseline_score,mean_baseline_score,pages_with_opportunity,opportunity_rate_pct
0,0,27192,0.0,0.026731,1407,5.174316
1,1,36399,0.0,0.253712,12363,33.965219
2,2,74719,0.0,0.125577,8906,11.919324


In [15]:
# Check how the highest baseline-opportunity pages are distributed
# across the discovered clusters.

top_cutoff = baseline_df["baseline_score"].quantile(0.90)

baseline_df["top_10pct_opportunity"] = (
    baseline_df["baseline_score"] >= top_cutoff
)

top_opportunity_by_cluster = (
    baseline_df
    .groupby("cluster")
    .agg(
        pages=("content_hash_id", "count"),
        top_10pct_opportunity=("top_10pct_opportunity", "sum")
    )
    .reset_index()
)

top_opportunity_by_cluster["share_of_top_opportunities_pct"] = (
    100
    * top_opportunity_by_cluster["top_10pct_opportunity"]
    / top_opportunity_by_cluster["top_10pct_opportunity"].sum()
)

display(top_opportunity_by_cluster)

,cluster,pages,top_10pct_opportunity,share_of_top_opportunities_pct
0,0,27192,282,2.038751
1,1,36399,7162,51.778485
2,2,74719,6388,46.182765


In [16]:
from sklearn.metrics import adjusted_rand_score

# Fit an independent K=3 model only on the holdout data.
# ARI handles arbitrary cluster-number labels automatically.

holdout_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

independent_holdout_labels = holdout_kmeans.fit_predict(
    X_holdout_scaled
)

holdout_ari = adjusted_rand_score(
    holdout_labels,
    independent_holdout_labels
)

print(f"Holdout silhouette: {holdout_silhouette:.4f}")
print(f"Holdout ARI:        {holdout_ari:.4f}")

Holdout silhouette: 0.3907
Holdout ARI:        0.7306


In [17]:
# Combined model-vs-baseline summary
# The baseline score is a review-priority benchmark, not a clustering feature.

final_comparison = cluster_profile[[
    "cluster",
    "pages",
    "page_share_pct",
    "median_impressions",
    "median_clicks",
    "median_ctr",
    "median_avg_position",
    "median_position_volatility"
]].merge(
    baseline_cluster_summary[[
        "cluster",
        "median_baseline_score",
        "mean_baseline_score",
        "opportunity_rate_pct"
    ]],
    on="cluster",
    how="left"
)

display(
    final_comparison.sort_values("cluster")
)

,cluster,pages,page_share_pct,median_impressions,median_clicks,median_ctr,median_avg_position,median_position_volatility,median_baseline_score,mean_baseline_score,opportunity_rate_pct
0,0,27192,19.660184,59.0,0.0,0.000000,40.500000,22.582873,0.0,0.026731,5.174316
1,1,36399,26.316969,2460.0,6.0,0.282885,7.917966,3.400076,0.0,0.253712,33.965219
2,2,74719,54.022847,38.0,0.0,0.000000,7.500000,4.387071,0.0,0.125577,11.919324


### Cluster interpretation

The final K=3 solution produces three distinct performance archetypes based on the original March 2026 search-performance features.

**Cluster 0 — Low-visibility, unstable pages**

Cluster 0 contains 19.66% of the development pages. Its median impressions are 59, median CTR is 0%, median average position is 40.5, and median position volatility is 22.56. This is the weakest search-performance pattern in the clustering solution, combining low visibility with weak ranking and high volatility.

**Cluster 1 — High-visibility performance pages**

Cluster 1 contains 26.32% of pages and has substantially higher visibility than the other clusters. Its median impressions are 2,460, median clicks are 6, median CTR is 0.283%, and median average position is 7.92, with low position volatility of 3.40. This cluster also contains 33.97% of pages with a positive Week-4 baseline opportunity score and 51.78% of the top 10% baseline opportunities.

**Cluster 2 — Low-visibility, relatively well-ranked pages**

Cluster 2 is the largest group at 54.02% of pages. It has low median impressions (38) and zero median CTR, but its median average position is 7.50 and median position volatility is 4.39. This indicates pages that can rank relatively well when they receive search exposure but currently have limited observed visibility or click activity.

### Action interpretation

The clusters can support different review priorities:

- **Cluster 0:** investigate content quality, search intent alignment, indexing/visibility issues, and whether pages should be improved, consolidated, or deprioritized. The clustering alone does not establish which action is correct.
- **Cluster 1:** protect strong-performing pages while reviewing the high-concentration CTR opportunities identified by the Week-4 baseline.
- **Cluster 2:** investigate why pages with relatively good average positions receive little search exposure or clicks. Query coverage, demand, indexing, and SERP context should be reviewed before deciding on an action.

The Week-4 baseline and K-Means therefore serve different purposes. The baseline provides a simple review-priority score, while K-Means provides a multidimensional view of recurring performance patterns. The model does not replace the baseline; it adds context around which types of pages contain those opportunities.

### Error analysis and limitations

The clustering solution can misclassify pages that sit near the boundaries between archetypes because K-Means assigns every page to exactly one cluster. It also summarizes each page using aggregate March 2026 performance metrics and therefore does not observe query intent, page content quality, seasonality, conversion value, SERP features, or business context.

The zero median CTR in Clusters 0 and 2 also reflects the sparsity of click data and should not be interpreted as evidence that every page in those clusters has zero click potential.

The holdout silhouette of 0.3907 compared with 0.4017 on development data, together with a holdout ARI of 0.7306, indicates that the three-cluster structure transfers reasonably well to unseen clients. These results support using the clusters as descriptive decision-support archetypes, not as ground-truth labels or causal explanations.

In [18]:
# Compact evidence table used for the final interpretation

interpretation_table = pd.DataFrame({
    "cluster": [0, 1, 2],
    "archetype": [
        "Low-visibility, unstable pages",
        "High-visibility performance pages",
        "Low-visibility, relatively well-ranked pages"
    ],
    "recommended_review": [
        "Investigate visibility, intent, quality and consolidation signals",
        "Protect performance and review CTR opportunities",
        "Investigate limited visibility despite relatively good ranking"
    ]
})

display(
    interpretation_table.merge(
        final_comparison,
        on="cluster",
        how="left"
    )
)

,cluster,archetype,recommended_review,pages,page_share_pct,median_impressions,median_clicks,median_ctr,median_avg_position,median_position_volatility,median_baseline_score,mean_baseline_score,opportunity_rate_pct
0,0,"Low-visibility, unstable pages","Investigate visibility, intent, quality and co...",27192,19.660184,59.0,0.0,0.000000,40.500000,22.582873,0.0,0.026731,5.174316
1,1,High-visibility performance pages,Protect performance and review CTR opportunities,36399,26.316969,2460.0,6.0,0.282885,7.917966,3.400076,0.0,0.253712,33.965219
2,2,"Low-visibility, relatively well-ranked pages",Investigate limited visibility despite relativ...,74719,54.022847,38.0,0.0,0.000000,7.500000,4.387071,0.0,0.125577,11.919324


### Self-check

- [x] Every section above is filled with analysis and supporting code.
- [x] The notebook uses a client-grouped development/holdout split with zero client overlap.
- [x] Preprocessing is fitted only on development data and applied unchanged to holdout data.
- [x] Candidate K values were compared using silhouette score and cluster-size balance.
- [x] K=3 was selected based on separation, balance, and interpretability rather than metric maximization alone.
- [x] Holdout stability was checked using silhouette score and adjusted Rand index.
- [x] Cluster profiles were inspected using the original, untransformed features.
- [x] The Week-4 baseline was used as a transparent review-priority benchmark and was not used as a clustering feature.
- [x] No ground-truth archetype labels or supervised accuracy metrics were invented.
- [x] Cluster names were assigned only after inspecting the resulting profiles.
- [x] Error analysis documents boundary cases and important limitations of aggregate metric clustering.
- [x] The interpretation uses descriptive and measured language rather than causal claims.
- [x] No client names, URLs, private queries, future-window data, or label-derived inputs were used.
- [x] The notebook is saved under `work/notebooks/w05_model.ipynb`.